In [1]:
! pip install -e .
! pip install lifelines


Obtaining file:///home/jovyan/Precipitation_paper
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for precip_helper (pyproject.toml) ... done
  Created wheel for precip_helper: filename=precip_helper-0.1.0-0.editable-py3-none-any.whl size=1296 sha256=e3a9811d01be2eb700583e544a28fed07d514cf563e9bc30a3876322c67375cc
  Stored in directory: /tmp/pip-ephem-wheel-cache-dwz0t_hr/wheels/85/d6/fd/4a136e03179e7b1256a36ee322e2837d14b136d70771439837
Successfully built precip_helper
  Attempting uninstall: precip_helper
    Found existing installation: precip_helper 0.1.0
    Uninstalling precip_helper-0.1.0:
      Successfully uninstalled precip_helper-0.1.0


In [2]:
import xarray as xr
import earthaccess
import boto3
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import warnings
from IPython.display import display, Markdown
import pandas as pd
import geopandas as gpd
import rasterio
import datetime
import pyarrow as pa
import pyarrow.parquet as pq
import os
import numpy as np
from lifelines import CoxPHFitter


warnings.filterwarnings('ignore')
%matplotlib inline

%load_ext autoreload
%autoreload 2


from precip_helper.utils import read_in_fires_and_precip

In [3]:
fires = read_in_fires_and_precip()

In [4]:
fires = fires[fires.l1_ecoregion != "GREAT PLAINS"]

In [5]:
### Reformatting the DF to be in the shape expected by lifelines. 

def get_cox_format(df):
    
    af_mask =  (df['start_off_12hrs'] >= 0) & (df['end_off_12hrs'] <= 0)
    new_df = pd.DataFrame({"UfireID" : df.UfireID.unique(), 
                          "duration" : df.duration.max(),
                          "ended": 1,
                          "precip_during_fire": df[af_mask].precipitation.sum(), 
                          "precip_post_fire": df[df.end_off_12hrs == 1].precipitation.unique(), 
                          "farea_final": df.farea.max(),
                          "n_pixels": df.n_pixels.max(),
                          "GACCAbbrev":  df.GACCAbbrev.unique(), 
                         'l1_ecoregion': df.l1_ecoregion.unique(), 
                         'number_of_rain_events': len(df[af_mask & (df.precipitation > 0)].precipitation)})

    return(new_df)

reg_fr = fires.groupby("UfireID").apply(get_cox_format).reset_index(drop = True)

reg_fr.loc[:, "total_precip"] = reg_fr.precip_post_fire + reg_fr.precip_during_fire


#### cph = CoxPHFitter()
# df must contain the covariates, the duration (time), and the event observed (boolean/int)

cph.fit(reg_fr[['duration', 'ended', 'number_of_rain_events']], duration_col='duration', event_col='ended')
cph.print_summary()
cph.plot()

In [6]:
cph.plot_partial_effects_on_outcome(covariates= "number_of_rain_events", values = [0, 1, 5, 10, 400])

NameError: name 'cph' is not defined

# Haha, looks like becuase precip varies through time, we are only capturing the effect of more time = more precip. Need to implement a time-varying related model

In [ ]:
## get the data into the weird format first. I followed https://lifelines.readthedocs.io/en/latest/Time%20varying%20survival%20regression.html

foo = fires[fires.UfireID == "10047_2022"]

## can the stop be equal to the start????? Lets find out! 

def cox_time_vary_format(df): # cuts off pre-fire precip, but includes post-fire precip 12 hour increment

    post_ig =  (df['start_off_12hrs'] >= 0) & (df.end_off_12hrs < 2) ## we could include more here. 
    df.loc[post_ig & (df.area_growth_at_t_km2.isna()), "area_growth_at_t_km2"] = 0 # include the real zero at the 12 hour mark, negated by the end event?
    end_marker = df[df.end_off_12hrs == 1].start_off_12hrs.values[0]

    
    new_df = pd.DataFrame({ 
                          "start" : df[post_ig].start_off_12hrs,
                          "stop":  df[post_ig].start_off_12hrs + 1,
                          "precipitation": df[post_ig].precipitation,
        
                          "area_growth_at_t_km2": df[post_ig].area_growth_at_t_km2,
                          #n_pixels": df.n_pixels,
                          #"GACCAbbrev":  df.GACCAbbrev.unique(), 
                         #'number_of_rain_events': len(df[post_ig & (df.precipitation > 0)].precipitation), # not time varying right now. 
        
                          })
    new_df.loc[:, "precip_cumsum"] = df[post_ig].precipitation.cumsum()
    new_df.loc[:, 'l1_ecoregion'] = df.l1_ecoregion.unique()[0]
    new_df.loc[:, "UfireID"] =  df.UfireID.unique()[0]
    new_df.loc[:, "GACCAbbrev"] = df.GACCAbbrev.unique()[0]
    new_df.loc[:, "ended"] = 0
    new_df.loc[new_df.start == end_marker, "ended"] = 1

    return(new_df)
    
    

In [ ]:
tmp = cox_time_vary_format(foo)

In [ ]:
fr_tv = fires.groupby("UfireID").apply(cox_time_vary_format).reset_index(drop = True)

In [ ]:
from lifelines import CoxTimeVaryingFitter

ctv = CoxTimeVaryingFitter()
ctv.fit(fr_tv[['start', 'stop', 'precipitation', 'UfireID', 'ended']], id_col='UfireID', event_col="ended", start_col="start", stop_col="stop", show_progress=True)
ctv.print_summary()
ctv.plot()

### Thoughts

- do I need to implment a formal equation (mabe using the *formula* parameter) to describe the non-linear relationship between precip and expected extinguishing / the non-linearity of precip itself? I am unclear on if letting precip time-vary accounts for it's non linearity. I am also not quite sure how to interpret the parameter. Should I jsut use a base level model with a transform (ie do we expect that the amount of precip is propotional to time, and therefore I could tansform the time bit out and get the precip effect?)
- check if number of precip events > effect than amount of precip, or close.
- Do all the assumption checks and critical thinking
- include co-variates?
- wow fireline stuff
- make fit and test datasets.
- could add percent containment!!!
- 

Fun stuff!!!